# FlyRank ML Capstone — CTR / Engagement Opportunity Scoring

**Project:** Search Visibility & CTR Opportunity Scoring

**Objective:** Build a reproducible, decision-support model that prioritizes content records showing meaningful visibility with comparatively low CTR.

**Important framing:** This analysis is observational and prioritization-oriented. It does not prove Google's ranking algorithm, establish causality, or guarantee traffic/CTR improvement.


## 1. Data and public-safe scope

Source file: `content_refresh_anonymized.csv`.

The dataset contains anonymized content-level search/content-performance records. Client identifiers and internal outcome/flag columns are excluded from modeling. Identifier fields are retained only where needed for grouping or final record-level ranking.

The capstone requires leakage checks and honest framing, so the model does **not** use existing outcome/decision flags such as `needs_ctr_fix`, `is_quick_win`, `is_underperformer`, or `is_initial_refresh_candidate`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("content_refresh_anonymized.csv")

print("Shape:", df.shape)
display(df.head())


In [ ]:
# Basic audit
print("Duplicate rows:", df.duplicated().sum())
print("Missing cells:", int(df.isna().sum().sum()))
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include="all").T.head(20))


## 2. Leakage-aware target definition

The opportunity label is intentionally defined from the outcome variables that we are trying to prioritize:

- high visibility: impressions at or above the 75th percentile
- low but non-zero CTR: CTR at or below the median among pages with non-zero CTR

The outcome columns used to define the label are removed from model features. This prevents the model from simply learning the answer from the target itself.


In [ ]:
impression_threshold = df["impressions_90d"].quantile(0.75)
nonzero_ctr = df.loc[df["ctr"] > 0, "ctr"]
ctr_threshold = nonzero_ctr.quantile(0.50)

df["opportunity"] = (
    (df["impressions_90d"] >= impression_threshold) &
    (df["ctr"] > 0) &
    (df["ctr"] <= ctr_threshold)
).astype(int)

print("Impression threshold:", impression_threshold)
print("Non-zero CTR median:", ctr_threshold)
print(df["opportunity"].value_counts())
print("Opportunity rate:", df["opportunity"].mean())


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["ctr"].clip(upper=df["ctr"].quantile(.99)), bins=40)
plt.xlabel("CTR (99th percentile capped for visualization)")
plt.ylabel("Pages")
plt.title("CTR Distribution")
plt.show()


## 3. Feature selection

Excluded:

- identifiers (`content_id`, `client_id`)
- direct target/outcome fields (`ctr`, `impressions_90d`)
- downstream aggregate outcomes and recent-window fields
- pre-existing decision flags

`client_id` is used only as a grouping variable for the train/test split, not as a feature.


In [ ]:
drop_cols = [
    "content_id", "client_id", "ctr", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "impressions_last_30d",
    "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "trend_pct", "health_score",
    "needs_indexing", "is_quick_win", "needs_ctr_fix", "needs_engagement_fix",
    "ai_opportunity", "is_underperformer", "is_declining",
    "is_initial_refresh_candidate"
]

X = df.drop(columns=drop_cols + ["opportunity"])
y = df["opportunity"]

categorical_cols = [c for c in X.columns if X[c].dtype == "object" or X[c].dtype == "bool"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Number of model features:", X.shape[1])
print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)


## 4. Grouped validation

The dataset contains multiple client groups. To reduce the chance that the same client's patterns appear in both training and testing, the split is grouped by `client_id`.

Random state is fixed at 42 for reproducibility.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train opportunity rate:", y_train.mean())
print("Test opportunity rate:", y_test.mean())


In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

lr = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ))
])

lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
prob_lr = lr.predict_proba(X_test)[:, 1]


In [ ]:
rf_preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

rf = Pipeline([
    ("preprocess", rf_preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]


In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

results = pd.DataFrame([
    ["Majority baseline",
     accuracy_score(y_test, pred_dummy),
     precision_score(y_test, pred_dummy, zero_division=0),
     recall_score(y_test, pred_dummy, zero_division=0),
     f1_score(y_test, pred_dummy, zero_division=0),
     np.nan],
    ["Logistic Regression",
     accuracy_score(y_test, pred_lr),
     precision_score(y_test, pred_lr, zero_division=0),
     recall_score(y_test, pred_lr, zero_division=0),
     f1_score(y_test, pred_lr, zero_division=0),
     roc_auc_score(y_test, prob_lr)],
    ["Random Forest",
     accuracy_score(y_test, pred_rf),
     precision_score(y_test, pred_rf, zero_division=0),
     recall_score(y_test, pred_rf, zero_division=0),
     f1_score(y_test, pred_rf, zero_division=0),
     roc_auc_score(y_test, prob_rf)]
], columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"])

display(results)


In [ ]:
plt.figure(figsize=(8,5))
plt.bar(results["Model"], results["F1"])
plt.ylabel("F1 score")
plt.title("Model vs Majority Baseline")
plt.xticks(rotation=15)
plt.show()


## 5. Final scoring and ranked recommendations

The final Logistic Regression model is retrained on all available records and used as a ranking engine. The ranking is decision support: high probability means the record resembles the defined opportunity profile.

Reason codes are descriptive and should be treated as review prompts, not guaranteed fixes.


In [ ]:
final_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ))
])

final_model.fit(X, y)
df["opportunity_probability"] = final_model.predict_proba(X)[:, 1]

ranked = df[df["opportunity"] == 1].copy()
ranked = ranked.sort_values("opportunity_probability", ascending=False)

ranked["reason_code"] = np.where(
    (ranked["avg_position"] <= 10) & (ranked["ctr"] <= ctr_threshold),
    "LOW_CTR + STRONG_POSITION",
    "LOW_CTR + HIGH_VISIBILITY"
)

display(ranked[[
    "content_id", "content_type", "main_intent", "search_volume",
    "avg_position", "ctr", "impressions_90d",
    "content_age_days", "days_since_last_update",
    "opportunity_probability", "reason_code"
]].head(25))


## 6. Recommended action playbook

1. Review high-visibility / low-CTR records for title and snippet-message alignment.
2. For pages with strong average position but low CTR, prioritize SERP messaging review.
3. For pages with weaker positions and high visibility, review intent alignment and content usefulness.
4. Monitor the ranked set after changes; do not interpret the score as a guaranteed traffic forecast.
5. Re-run the pipeline when a new dataset release is available.
